# 📚 AI Agents Workshop · Level 1 Fundamentals · Day 2
## Building Your Own AI Agent: BrightPage Stationery & Office Supplies

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

### 🎯 Day 2 Overview: From a Single API Call to an Interactive Conversational Agent
On **Day 1**, we learned how LLMs predict tokens, how temperature controls variability, how token budgets work, and how to execute a raw API call using Groq.

Today on **Day 2**, we take the critical leap: we build a complete, stateful AI Assistant tailored for **BrightPage Stationery & Office Supplies**!

#### 🧭 What We Build & Master Today:
1. **Messages & Roles:** How conversation scripts are built with `system`, `user`, and `assistant`.
2. **The 4-Pillar System Prompt:** Calibrating **Role**, **Facts**, **Rules**, and **Format** to stop hallucinations and enforce shop policies.
3. **Few-Shot Prompting:** Training the model in-context with examples to classify customer intents (`order`, `inquiry`, `complaint`).
4. **Structured JSON Output:** Extracting messy WhatsApp/chat messages into clean, database-ready order JSON.
5. **Memory by Hand (Chat Loop):** Managing conversation history across turns so the agent remembers customer names and requests.
6. **Context Window Management:** Implementing sliding-window history trimming (`MAX_TURNS`) to control token consumption.
7. **Python for Agents:** Dictionaries, type hints, docstrings, and `@log_call` decorators (the building blocks for Day 3 & 4 tool calling!).
8. **Error Resilience (`try / except`):** Catching `RateLimitError` and connection issues to provide polite fallbacks instead of crashing.
9. **Interactive Colab Dashboard:** A visual UI using `ipywidgets` to test chat, intent classification, and order parsing directly in your notebook.
10. **Student's 10% Verification Suite:** Real tests evaluating temperature differences, unlisted item boundary checks, and memory limits.


---
### 🛠️ Step 1: Install Required Libraries
We install `openai` (for the standard Groq-compatible client), `langchain_groq` (ready for Day 3), `python-dotenv` (for loading `.env` locally), and `ipywidgets` (for our interactive dashboard).


In [ ]:
# Install dependencies (runs once per Colab session)
!pip install -q openai langchain_groq python-dotenv ipywidgets


---
### 🔑 Step 2: Configure Your API Key (Same Key as Day 1!)
> **💡 Key Clarification:**
> **You can use the exact same `GROQ_API_KEY` that you used for Day 1!**
> There is no need to generate or use a different key.
> 
> - **In Google Colab:** Click the **🔑 Key icon (Secrets)** on the left sidebar $\rightarrow$ Ensure `GROQ_API_KEY` is present $\rightarrow$ Toggle **"Notebook access"** ON.
> - **In Local / VS Code:** The key is automatically read from `DAY2/.env` or `DAY1/.env` via `python-dotenv`.

Let's load the key, verify connectivity, and initialize our clients.


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from langchain_groq import ChatGroq

# 1. Automatically locate and load the single root .env file (for local runs)
load_dotenv(find_dotenv())

# 2. Attempt loading from Google Colab Secrets (for Colab runs)
if not os.environ.get("GROQ_API_KEY"):
    try:
        from google.colab import userdata
        os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
        print("✅ Successfully retrieved GROQ_API_KEY from Google Colab Secrets!")
    except Exception:
        pass

api_key = os.environ.get("GROQ_API_KEY")
if not api_key:
    raise ValueError("❌ GROQ_API_KEY is missing! Please configure it in Colab Secrets or in the root .env file.")

# Model selection: openai/gpt-oss-120b (high capacity) or openai/gpt-oss-20b
MODEL_NAME = "openai/gpt-oss-120b"

# Initialize standard OpenAI client pointing to Groq's high-speed API
client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

# Initialize LangChain ChatGroq instance (ready for Day 3 agent frameworks)
llm = ChatGroq(
    model=MODEL_NAME,
    temperature=0.2,
    groq_api_key=api_key
)

print(f"🚀 Groq client connected successfully! Target Model: {MODEL_NAME}")


In [ ]:
# Sanity Check: Test API response
test_response = client.chat.completions.create(
    model=MODEL_NAME,
    temperature=0.2,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Respond with 'BrightPage Stationery Agent is Ready!'"}
    ]
)
print("API Response:", test_response.choices[0].message.content.strip())


---
### 🏪 Step 3: The System Prompt (Role · Facts · Rules · Format)
The system prompt is the foundational job description given to the model.
We structure it using the **4 Pillars**:
- **ROLE:** Virtual assistant for *BrightPage Stationery & Office Supplies*.
- **FACTS:** Inventory price list, store operating hours, free delivery eligibility (above ₹500), same-day delivery cutoff (3:00 PM), and bulk discount (10% on 10+ items).
- **RULES:** Never promise unlisted items; if an item is not in stock, reply *"I'll check with the store manager about availability and get back to you shortly."* Never promise delivery outside town.
- **FORMAT:** Concise, friendly, under 60 words, in English or customer's language.


In [ ]:
import json

# Inventory Catalogue & Pricing in Indian Rupees (INR)
STATIONERY_CATALOGUE = {
    "spiral notebook (200 pgs)": 85,
    "a4 copier paper ream (500 sheets)": 320,
    "gel pen pack (10 pens, blue/black)": 150,
    "pastel highlighter set (6 colors)": 180,
    "heavy-duty stapler + pins": 240,
    "sticky notes pack (400 sheets)": 95,
    "geometry box set": 160,
    "whiteboard marker set (4 colors)": 120,
    "permanent marker (black)": 35,
    "clear file folder (pack of 5)": 110
}

# The 4-Pillar System Prompt
SYSTEM_PROMPT = f"""ROLE:
You are the friendly, helpful AI virtual assistant for 'BrightPage Stationery & Office Supplies'.

FACTS:
- Store Catalogue & Prices (INR): {json.dumps(STATIONERY_CATALOGUE)}
- Operating Hours: Mon-Sat 9:00 AM - 8:30 PM, Sunday 10:00 AM - 4:00 PM.
- Delivery Policy: Free local home/office delivery on orders above ₹500. Standard ₹40 delivery fee for orders below ₹500.
- Delivery Timing: Orders placed before 3:00 PM delivered same evening; orders after 3:00 PM delivered next morning.
- Bulk Discount: 10% discount automatically applied on orders of 10+ identical items or order total exceeding ₹1,000.
- Return/Exchange Policy: Defective or damaged items replaced within 3 days with receipt.

RULES:
- Never promise delivery outside our local city limits.
- If an item is NOT in our FACTS catalogue (e.g. laptops, heavy furniture, textbooks), say:
  "I'll check with the store manager about availability and get back to you shortly."
- Never invent unlisted products or guess unverified delivery times.
- Always confirm item quantities, total pricing, and delivery preference before finalizing an order.
- Maintain a warm, courteous tone at all times.

FORMAT:
- Keep responses friendly, clear, and under 60 words.
- Match the customer's language (English, Telugu, or Hinglish if requested)."""

print("✅ System prompt defined successfully.")


---
### 🎯 Step 4: Few-Shot Prompting (Customer Intent Classifier)
Instead of asking the model to classify in an open-ended format (zero-shot), we provide **3 clear worked examples** (few-shot).
This forces the model to adhere strictly to our desired output shape: **exactly one lowercase word** (`order`, `inquiry`, or `complaint`), making downstream code branches reliable.


In [ ]:
def classify_intent(customer_message: str) -> str:
    """Classifies customer message into 'order', 'inquiry', or 'complaint' using few-shot prompting."""
    messages = [
        {
            "role": "system",
            "content": "You are a customer intent classifier for BrightPage Stationery. "
                       "Respond with EXACTLY ONE word from these three options: order, inquiry, or complaint. "
                       "Do not write any punctuation or explanation."
        },
        # Few-shot example 1: Order
        {"role": "user", "content": "Can I get 5 reams of A4 paper delivered to Tech Park tomorrow morning?"},
        {"role": "assistant", "content": "order"},
        
        # Few-shot example 2: Inquiry
        {"role": "user", "content": "Do you have Classmate spiral notebooks with grid lines?"},
        {"role": "assistant", "content": "inquiry"},
        
        # Few-shot example 3: Complaint
        {"role": "user", "content": "The gel pen pack I bought yesterday is leaking ink everywhere!"},
        {"role": "assistant", "content": "complaint"},
        
        # Actual incoming query
        {"role": "user", "content": customer_message}
    ]
    
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,  # Temperature 0.0 for consistent classification
        messages=messages
    )
    return response.choices[0].message.content.strip().lower()

# Test the few-shot classifier with diverse queries
test_queries = [
    "I want to order 3 packs of pastel highlighters and 2 sticky note pads.",
    "What are your store opening hours on Sunday?",
    "The stapler pins you gave me don't fit the stapler model!",
    "Do you offer free delivery for orders around 600 rupees?"
]

print("📊 Few-Shot Intent Classification Results:")
for q in test_queries:
    label = classify_intent(q)
    print(f"  • "{q}" \n    ➔ [{label.upper()}]\n")


---
### 📝 Step 5: Structured Output (Extracting Clean JSON Orders)
Customers type unstructured, conversational messages (*"Hi this is Ramesh, please send 2 reams of A4 paper and 3 pastel highlighters to my clinic urgently"*).
To book orders into an inventory sheet or database, we need **structured JSON**. We prompt the model with an exact schema and parse it with `json.loads()`.


In [ ]:
import re

ORDER_EXTRACTION_SYSTEM = """You are an order extraction engine for BrightPage Stationery.
Extract customer order details from the message into a strictly valid JSON object.
Return ONLY raw JSON. No markdown backticks, no code fences, no conversational filler.

JSON Schema:
{
  "customer_name": string or null,
  "items": [
    {
      "item_name": string,
      "quantity": integer
    }
  ],
  "delivery_type": "home_delivery" or "store_pickup",
  "urgency": "normal" or "urgent",
  "notes": string or null
}"""

def extract_order_json(customer_message: str) -> dict:
    """Extracts structured order dictionary from conversational customer text."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.0,
        messages=[
            {"role": "system", "content": ORDER_EXTRACTION_SYSTEM},
            {"role": "user", "content": customer_message}
        ]
    )
    raw_content = response.choices[0].message.content.strip()
    
    # Strip markdown fences if the model wrapped output in ```json ... ```
    cleaned = re.sub(r"^```json\s*|^```\s*|\s*```$", "", raw_content, flags=re.MULTILINE).strip()
    
    try:
        order_dict = json.loads(cleaned)
        return order_dict
    except json.JSONDecodeError as err:
        print(f"⚠️ JSON parsing error: {err}. Raw output was:\n{raw_content}")
        return {"error": "Failed to parse JSON", "raw": raw_content}

# Test order extraction
sample_order_msg = "Hello! This is Ramesh Kumar. Please send 2 reams of A4 copier paper and 3 pastel highlighter sets to my clinic urgently."
extracted = extract_order_json(sample_order_msg)

print("📦 Extracted Order Dictionary:")
print(json.dumps(extracted, indent=2))


---
### 🐍 Step 6: Python Foundations for Agents (Catalogue, Types & Decorators)
Before agents can call functions autonomously on Day 4, we must establish:
1. **Data in Dictionaries:** Fast lookup and safe access via `.get()`.
2. **Type Hints & Docstrings:** Clear contracts (`item: str, qty: int -> dict`). The docstring is the instruction manual the agent reads to understand the tool.
3. **Decorators (`@log_call`):** Wrapping functions to trace arguments, results, and execution time without modifying the function's internal code.


In [ ]:
import functools
import time

def log_call(func):
    """Decorator that logs function execution, arguments, and return values."""
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        arg_str = ", ".join([repr(a) for a in args] + [f"{k}={v!r}" for k, v in kwargs.items()])
        print(f"   ⚡ [LOG CALL] Entering {func.__name__}({arg_str})")
        start = time.perf_counter()
        result = func(*args, **kwargs)
        duration_ms = (time.perf_counter() - start) * 1000
        print(f"   ⚡ [LOG RETURN] {func.__name__} returned {result!r} ({duration_ms:.2f} ms)")
        return result
    return wrapper

@log_call
def calculate_item_total(item_name: str, qty: int = 1) -> dict:
    """
    Calculate the total cost for a specific stationery item and quantity.
    Applies a 10% bulk discount if quantity is 10 or more.
    
    Args:
        item_name: The name of the item from the stationery catalogue.
        qty: The quantity requested (default is 1).
        
    Returns:
        dict: A billing breakdown with item, unit_price, qty, discount, and final_total.
    """
    cleaned_name = item_name.lower().strip()
    
    # Safe lookup using catalogue dictionary
    unit_price = STATIONERY_CATALOGUE.get(cleaned_name)
    if unit_price is None:
        # Partial match fallback
        for cat_item, price in STATIONERY_CATALOGUE.items():
            if cleaned_name in cat_item or cat_item in cleaned_name:
                unit_price = price
                cleaned_name = cat_item
                break
                
    if unit_price is None:
        return {"error": f"Item '{item_name}' is not currently in the catalogue."}
        
    subtotal = unit_price * qty
    discount = (subtotal * 0.10) if qty >= 10 else 0.0
    final_total = subtotal - discount
    
    return {
        "item": cleaned_name,
        "unit_price": unit_price,
        "quantity": qty,
        "subtotal": subtotal,
        "bulk_discount": discount,
        "final_total": final_total
    }

# Test tool function with decorator logging
print("Testing calculate_item_total with logging:")
calc_result = calculate_item_total("spiral notebook (200 pgs)", 12)
print("Result:", calc_result)


---
### 🧠 Step 7: Multi-Turn Memory & Resilient Chat Loop
As demonstrated on Day 1, **LLMs are completely stateless**. They remember nothing between calls.
To create a real stationery assistant that remembers:
1. We keep a `history` list of all user prompts and assistant replies.
2. We **trim older turns** (`MAX_TURNS`) so the conversation stays within context limits and doesn't waste tokens.
3. We wrap API calls in `try / except` to catch `RateLimitError` or network dropouts and reply politely rather than crashing.


In [ ]:
from openai import RateLimitError, APIConnectionError, AuthenticationError

class StationeryAgent:
    """Conversational Agent for BrightPage Stationery with stateful memory and error safety."""
    
    def __init__(self, system_prompt: str, max_turns: int = 8, model: str = MODEL_NAME):
        self.system_prompt = system_prompt
        self.max_turns = max_turns
        self.model = model
        self.history = [{"role": "system", "content": self.system_prompt}]
        self.total_tokens_used = 0
        
    def reset(self):
        """Reset memory to initial system prompt state."""
        self.history = [{"role": "system", "content": self.system_prompt}]
        print("🔄 Agent memory cleared. Started new session.")
        
    def chat(self, user_message: str) -> str:
        """Send message to agent, retain history, and return assistant reply."""
        if not user_message.strip():
            return "Please type a message!"
            
        # 1. Append user message to history
        self.history.append({"role": "user", "content": user_message})
        
        # 2. Context Window Management: Keep system prompt + last N turns
        # Each turn is 2 messages (1 user + 1 assistant)
        trimmed_messages = [self.history[0]] + self.history[1:][-(self.max_turns * 2):]
        
        # 3. Call API within error safety net (try/except)
        try:
            response = client.chat.completions.create(
                model=self.model,
                messages=trimmed_messages,
                temperature=0.3
            )
            reply = response.choices[0].message.content.strip()
            
            # Track token usage
            if hasattr(response, "usage") and response.usage:
                self.total_tokens_used += response.usage.total_tokens
                
        except RateLimitError:
            reply = "We are currently experiencing high customer volume. Please wait a moment and try again!"
        except APIConnectionError:
            reply = "I cannot reach the store servers right now. Please check your internet connection."
        except AuthenticationError:
            reply = "Authentication error: Store API key is misconfigured. Please notify the manager."
        except Exception as e:
            reply = f"An unexpected error occurred: {str(e)}. I'll notify the store owner."
            
        # 4. Append assistant response to memory
        self.history.append({"role": "assistant", "content": reply})
        return reply

# Create the agent instance
stationery_agent = StationeryAgent(system_prompt=SYSTEM_PROMPT, max_turns=6)
print("✅ StationeryAgent ready for multi-turn conversations.")


#### Test Multi-Turn Memory Verification
Let's simulate a 3-turn conversation where the customer gives their name in turn 1, asks about items in turn 2, and asks the agent to recall who they are in turn 3.


In [ ]:
# Turn 1: Introduce customer name
print("Customer: Hi, I'm Ananya. Do you have heavy duty staplers in stock?")
r1 = stationery_agent.chat("Hi, I'm Ananya. Do you have heavy duty staplers in stock?")
print(f"Agent: {r1}\n")

# Turn 2: Ask about delivery & price
print("Customer: How much is it, and is delivery free if I also take an A4 paper ream?")
r2 = stationery_agent.chat("How much is it, and is delivery free if I also take an A4 paper ream?")
print(f"Agent: {r2}\n")

# Turn 3: Test memory recall
print("Customer: Great! By the way, what was my name again?")
r3 = stationery_agent.chat("Great! By the way, what was my name again?")
print(f"Agent: {r3}\n")

print(f"Total tokens used in this multi-turn session: {stationery_agent.total_tokens_used}")


---
### 💻 Step 8: Interactive UI Dashboard (Google Colab / Jupyter)
Run the cell below to launch the **BrightPage Stationery Assistant Dashboard**.
You can:
- **Chat interactively** in real time with styled user and bot bubbles.
- Click **Quick Presets** to test delivery rules, bulk discounts, and unlisted items.
- Switch tabs to test the **Few-Shot Intent Classifier** or the **JSON Order Extractor**!


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Create fresh agent instance for the interactive UI
ui_agent = StationeryAgent(system_prompt=SYSTEM_PROMPT, max_turns=8)

# 1. UI Components
header_html = widgets.HTML("""
<div style="background: linear-gradient(135deg, #1e3c72 0%, #2a5298 100%); padding: 16px 20px; border-radius: 10px; color: white; margin-bottom: 12px;">
    <h2 style="margin: 0 0 6px 0; font-family: 'Segoe UI', Tahoma, sans-serif;">✏️ BrightPage Stationery Assistant</h2>
    <p style="margin: 0; opacity: 0.9; font-size: 14px;">Day 2 Interactive Agent: Memory, Order Parsing & Policy Enforcement</p>
</div>
""")

# Chat output area
chat_display = widgets.Output(layout=widgets.Layout(height="340px", overflow="auto", border="1px solid #dcdcdc", padding="12px", border_radius="8px", background_color="#fcfcfc"))

# Chat text input & send button
chat_input = widgets.Text(
    value="",
    placeholder="Type your message (e.g., 'Hi, I need 5 notebooks, do you deliver?')...",
    layout=widgets.Layout(width="78%")
)
send_btn = widgets.Button(description="Send", button_style="primary", icon="paper-plane", layout=widgets.Layout(width="10%"))
reset_btn = widgets.Button(description="Reset Chat", button_style="warning", icon="refresh", layout=widgets.Layout(width="10%"))

# Preset query buttons
btn_preset1 = widgets.Button(description="Preset: Free Delivery Policy", layout=widgets.Layout(width="auto"))
btn_preset2 = widgets.Button(description="Preset: Bulk Discount Check", layout=widgets.Layout(width="auto"))
btn_preset3 = widgets.Button(description="Preset: Unlisted Item Test", layout=widgets.Layout(width="auto"))

def render_conversation():
    with chat_display:
        clear_output()
        for msg in ui_agent.history:
            role = msg["role"]
            content = msg["content"]
            if role == "system":
                continue
            elif role == "user":
                display(HTML(f"""
                <div style="display: flex; justify-content: flex-end; margin-bottom: 8px;">
                    <div style="background-color: #007bff; color: white; padding: 8px 14px; border-radius: 14px 14px 2px 14px; max-width: 75%; font-family: 'Segoe UI', sans-serif;">
                        <b>You:</b> {content}
                    </div>
                </div>
                """))
            elif role == "assistant":
                display(HTML(f"""
                <div style="display: flex; justify-content: flex-start; margin-bottom: 8px;">
                    <div style="background-color: #e9ecef; color: #212529; padding: 8px 14px; border-radius: 14px 14px 14px 2px; max-width: 75%; font-family: 'Segoe UI', sans-serif;">
                        <b>BrightPage Bot:</b> {content}
                    </div>
                </div>
                """))

def on_send_clicked(b):
    user_text = chat_input.value.strip()
    if not user_text:
        return
    chat_input.value = ""
    render_conversation()
    with chat_display:
        display(HTML("<p style='color: #888;'><i>BrightPage Bot is typing...</i></p>"))
    ui_agent.chat(user_text)
    render_conversation()

def on_reset_clicked(b):
    ui_agent.reset()
    render_conversation()

send_btn.on_click(on_send_clicked)
reset_btn.on_click(on_reset_clicked)
chat_input.on_submit(on_send_clicked)

btn_preset1.on_click(lambda b: (setattr(chat_input, 'value', "What is your minimum order for free delivery?"), on_send_clicked(None)))
btn_preset2.on_click(lambda b: (setattr(chat_input, 'value', "Can I get a discount if I buy 15 geometry boxes?"), on_send_clicked(None)))
btn_preset3.on_click(lambda b: (setattr(chat_input, 'value', "Do you sell Apple iPad stylus pens and 3D printing filaments?"), on_send_clicked(None)))

# Assemble Chat Tab
chat_row = widgets.HBox([chat_input, send_btn, reset_btn])
preset_row = widgets.HBox([widgets.Label("💡 Quick Presets:"), btn_preset1, btn_preset2, btn_preset3])
chat_tab = widgets.VBox([chat_display, chat_row, preset_row])

# 2. Intent Classifier Tab
intent_input = widgets.Text(value="Can you deliver 10 reams of A4 paper by 2 PM today?", layout=widgets.Layout(width="80%"))
intent_btn = widgets.Button(description="Classify Intent", button_style="info", icon="tag")
intent_output = widgets.Output()

def on_intent_clicked(b):
    with intent_output:
        clear_output()
        display(HTML("<i>Classifying...</i>"))
        res = classify_intent(intent_input.value)
        color = "#28a745" if res == "order" else "#007bff" if res == "inquiry" else "#dc3545"
        clear_output()
        display(HTML(f"""
        <div style="padding: 12px; border-left: 4px solid {color}; background-color: #f8f9fa; border-radius: 6px; margin-top: 8px;">
            <b>Input Message:</b> {intent_input.value}<br>
            <b>Classified Intent:</b> <span style="background-color: {color}; color: white; padding: 3px 8px; border-radius: 4px; font-weight: bold; text-transform: uppercase;">{res}</span>
        </div>
        """))

intent_btn.on_click(on_intent_clicked)
intent_tab = widgets.VBox([widgets.Label("Test the few-shot intent classifier on any customer message:"), widgets.HBox([intent_input, intent_btn]), intent_output])

# 3. JSON Extractor Tab
json_input = widgets.Textarea(value="Hi, this is Vikram from Apex College. Need 8 packs of whiteboard markers and 20 spiral notebooks delivered to our admin office.", layout=widgets.Layout(width="90%", height="80px"))
json_btn = widgets.Button(description="Extract JSON Order", button_style="success", icon="code")
json_output = widgets.Output()

def on_json_clicked(b):
    with json_output:
        clear_output()
        display(HTML("<i>Extracting structured order...</i>"))
        res = extract_order_json(json_input.value)
        clear_output()
        display(HTML(f"""
        <div style="background-color: #1e1e1e; color: #dcdcdc; padding: 12px; border-radius: 6px; font-family: monospace; white-space: pre-wrap; margin-top: 8px;">
{json.dumps(res, indent=2)}
        </div>
        """))

json_btn.on_click(on_json_clicked)
json_tab = widgets.VBox([widgets.Label("Test extracting unstructured customer message into strict JSON:"), json_input, json_btn, json_output])

# Tabs Container
tabs = widgets.Tab(children=[chat_tab, intent_tab, json_tab])
tabs.set_title(0, "💬 Live Stationery Chat")
tabs.set_title(1, "🏷️ Few-Shot Intent Classifier")
tabs.set_title(2, "📦 JSON Order Parser")

display(widgets.VBox([header_html, tabs]))
render_conversation()


---
### 🧪 Step 9: Student's 10% Verification Suite
The Day 2 tutorial concludes with the **Student's 10%**: hands-on verification tests where we systematically inspect agent behavior.

#### Experiment 1: Temperature Dial (Creativity vs Determinism)
We ask for a brand slogan for BrightPage Stationery at $T = 0.2$ (focused, repeatable) vs $T = 1.2$ (creative, varied).


In [ ]:
print("🔬 Experiment 1: Temperature Dial Test on Stationery Slogan\n")

prompt = "Write a catchy 6-word tagline for BrightPage Stationery & Office Supplies."

print("--- Low Temperature (T = 0.2) ---")
for i in range(3):
    res = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.2,
        messages=[{"role": "user", "content": prompt}]
    )
    print(f"Run {i+1}: {res.choices[0].message.content.strip()}")

print("\n--- High Temperature (T = 1.2) ---")
for i in range(3):
    res = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=1.2,
        messages=[{"role": "user", "content": prompt}]
    )
    print(f"Run {i+1}: {res.choices[0].message.content.strip()}")


#### Experiment 2: Hallucination Stress Test (Boundary Verification)
We test what happens when a customer asks for items NOT on the catalogue (e.g. 3D printer filament or laptop screens).
According to our **RULES**, the agent must state: *"I'll check with the store manager..."* and NOT invent prices or availability.


In [ ]:
print("🔬 Experiment 2: Hallucination Stress Test on Unstocked Items\n")

test_hallucination_queries = [
    "Do you sell resin 3D printer filaments?",
    "Can you deliver a 27-inch computer monitor to my office today?",
    "Do you have watercolor sketchbooks?"
]

eval_agent = StationeryAgent(system_prompt=SYSTEM_PROMPT, max_turns=2)

for q in test_hallucination_queries:
    reply = eval_agent.chat(q)
    print(f"Customer: {q}")
    print(f"Agent:    {reply}\n")


#### Experiment 3: Memory Trimming Verification
We verify what happens when `max_turns` is exceeded.
If `max_turns = 2`, information given 4 turns ago will be safely dropped to preserve the context window.


In [ ]:
print("🔬 Experiment 3: Memory Window Verification\n")

# Create an agent with a very small window of 2 turns
small_window_agent = StationeryAgent(system_prompt=SYSTEM_PROMPT, max_turns=2)

print("1. Turn 1: Customer gives their name")
r1 = small_window_agent.chat("My name is Suresh and my account number is BP-904.")
print("Bot:", r1, "\n")

print("2. Turn 2: Customer asks about notebooks")
r2 = small_window_agent.chat("What is the price of your spiral notebooks?")
print("Bot:", r2, "\n")

print("3. Turn 3: Customer asks about highlighters")
r3 = small_window_agent.chat("How many colors come in the pastel highlighter set?")
print("Bot:", r3, "\n")

print("4. Turn 4: Check if account number and name are still in memory")
r4 = small_window_agent.chat("What was my name and account number?")
print("Bot:", r4, "\n")

print("Active history message count in memory:", len(small_window_agent.history))


---
## 🏆 Summary: What You Built on Day 2

| Concept | Implementation in BrightPage Stationery Agent | Purpose |
| :--- | :--- | :--- |
| **Messages & Roles** | `system`, `user`, `assistant` | Structured conversation protocol |
| **System Prompt** | Role, Facts, Rules, Format | Firm grounding, prevents hallucinations & risky promises |
| **Few-Shot Prompting** | 3 categorized stationery query examples | Guarantees exact 1-word intent labels (`order`, `inquiry`, `complaint`) |
| **Structured Output** | Strict JSON extraction schema + regex cleaner | Transforms unstructured customer chat into database-ready orders |
| **Python Tools** | Dictionaries, type hints, docstrings | Prepares functions for autonomous tool calling |
| **Observability** | `@log_call` decorator | Real-time logging of inputs, outputs, and runtime |
| **Memory Loop** | `history` list + sliding window trimming | Enables natural conversation while protecting context window |
| **Error Handling** | `try / except RateLimitError` | Resilient, polite bot that never crashes on users |
| **Interactive UI** | Google Colab `ipywidgets` dashboard | Visual, testable user interface in the notebook |

---
### 🚀 Coming Up on Day 3: Turning Your Bot Into an Autonomous Agent
Tomorrow on **Day 3**, we introduce **LangChain**, the **ReAct (Reason + Act) loop**, and let the agent call our python functions (*tools*) dynamically whenever a calculation, stock check, or order logging is required!
